Pset 6

In [1]:
import pandas as pd
import numpy as np

In [2]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [3]:
import matplotlib.pyplot as plt

In [4]:
import statsmodels.formula.api as smf

In [5]:
import seaborn as sns

In [6]:
from linearmodels import IV2SLS


In [7]:
import scipy.io
mat = scipy.io.loadmat('../data/jacob_lefgren.mat')

In [8]:
data = pd.DataFrame(np.array([mat["fail"], mat["summer"], mat["rdge0"], mat["mtge0"],  
                              mat["rdge1"], mat["mtge1"], mat["rmath1"], mat["rread1"], 
                              mat["rmath0"], mat["rread0"], mat["male"], mat["black"], 
                              mat["cohort"]
                             ]).reshape([13, 70831])).transpose()

In [9]:
data.columns = ['fail', 'summer', 'rdge0', 'mtge0', 'rdge1', 'mtge1', 'rmath1', 'rread1',
                'rmath0', 'rread0', 'male', 'black', 'cohort'
               ]

a)

In [10]:
threshold = 2.75

In [11]:
variable = 'running'

In [12]:
outcome = 'rmath1'

In [13]:
treatment = 'summer'

In [14]:
data[variable] = data[['mtge0', 'rdge0']].min(axis = 1)

In [15]:
data['elegible'] = (data[variable] < threshold) * 1

In [16]:
data['elegible'].mean()

0.4221315525687905

In [17]:
data.groupby('elegible')['summer'].mean()

elegible
0    0.004056
1    0.936923
Name: summer, dtype: float64

Less than 1% of non-elegible students participates on summer school. More 90% of students who are elegible participates in summer schools. We need to estimate a fuzzy RDD considering that not all elegible participants received the treatment, but the increase in the probability of enrolling into summer school is higher given the threshold.

The interpretation of our results should be considered as a LATE, considering that the threshold is a string incentive to enroll into summer school. Therefore, the estimated effect should be interpreted as the effect on compliers, students who will attend summer school if they got less than 2.75 in any of the both scores, reading or math scores.

b)

Implementing the optimal bandwith estimator

In [18]:

def get_optimal_bandwith(data, variable, outcome, threshold):

    # Step 1
    
    N = data.shape[0]
    
    s2_X = data[variable].var()*N/(N - 1)
    
    h1 = 1.84 * s2_X ** 0.5 / (N **0.2)
    
    #Step 2
    
    sample_left = data[(data[variable] > threshold - h1) & (data[variable] < threshold)]
    sample_right = data[(data[variable] < threshold + h1) & (data[variable] > threshold)]
    
    bar_Y_left_h1 = sample_left[outcome].mean()
    bar_Y_right_h1 = sample_right[outcome].mean()
    
    N_h1_left = sample_left.shape[0]
    N_h1_right = sample_right.shape[0]
    
    f_c = (N_h1_left + N_h1_right) / (2 * N * h1)
    
    sigma2_left = sample_left[variable].var() * N_h1_left/(N_h1_left - 1)
    sigma2_right = sample_right[variable].var() * N_h1_right/(N_h1_right - 1)
    
    # Step 3
    
    
    
    model = smf.ols(f"{outcome} ~ 1 + I({variable} - {threshold}) + I(({variable} > {threshold})) + I(({variable} - {threshold}) ** 2) + I(({variable} - {threshold})** 3)", data = data).fit()   
    
    m3_c = model.params.iloc[-1]
    
    N_right = (data[variable] > threshold).sum()
    N_left = (data[variable] < threshold).sum()
    
    h2_right = 3.56 * (sigma2_right/(f_c * (m3_c ** 2))) ** (1/7) * (N_right ** (-1/7))
    
    h2_left = 3.56 * (sigma2_left/(f_c * (m3_c ** 2))) ** (1/7) * (N_left ** (-1/7))
    
    # Curvature estimation
    
    sample_left_h2 = data[(data[variable] < threshold) & (data[variable] > threshold - h2_left)]
    sample_right_h2 = data[(data[variable] > threshold) & (data[variable] < threshold + h2_right)]
    
    model_left = smf.ols(f"{outcome} ~ 1 + I({variable} - {threshold}) + I(({variable} - {threshold}) ** 2)", data = sample_left_h2).fit()
    model_right = smf.ols(f"{outcome} ~ 1 + I({variable} - {threshold}) + I(({variable} - {threshold}) ** 2)", data = sample_right_h2).fit()
    
    N2_right =sample_right_h2.shape[0]
    N2_left = sample_left_h2.shape[0]
    
    m2_right = 2 * model_right.params.iloc[-1]
    m2_left = 2 * model_left.params.iloc[-1]
    
    # Regularization
    
    r_right = 2160 * sigma2_right / (N2_right * (h2_right ** 4))
    
    r_left = 2160 * sigma2_left / (N2_left * (h2_left ** 4))
    
    # Optimal bandwith
    
    h_opt_edge = 3.4375 * ((sigma2_left + sigma2_right) / (f_c * ((m2_right - m2_left) ** 2) + r_left + r_right))**(1/5) * (N ** (-1/5))
    h_opt_unif = 5.40 * ((sigma2_left + sigma2_right) / (f_c * ((m2_right - m2_left) ** 2) + r_left + r_right))**(1/5) * (N ** (-1/5))

    return h_opt_edge, h_opt_unif

In [19]:
h_opt_edge, h_opt_unif = get_optimal_bandwith(data, variable, outcome, threshold)

In [20]:
h_opt = h_opt_unif

Estimating the fuzzy RDD considering an h_opt, for linear and quadratic specification

In [21]:
def estimate_frdd(data, variable, treatment, outcome, h_opt, threshold):

    m2s = IV2SLS.from_formula(f"{outcome} ~ 1 + [{treatment}~I({variable} > {threshold})] + I({variable} - {threshold}) + I({variable} - {threshold}):I({variable} > {threshold})", 
                          data = data[(data[variable] - threshold).apply(abs) < h_opt]).fit(cov_type = 'robust')

    m2s_q = IV2SLS.from_formula(f"{outcome} ~ 1 + [{treatment}~I({variable} > {threshold})] + I({variable} - {threshold}) + I({variable} - {threshold}):I({variable} > {threshold}) + I(({variable} - {threshold}) ** 2) + I(({variable} - {threshold})**2):I({variable} > {threshold})", 
                          data = data[(data[variable] - threshold).apply(abs) < h_opt]).fit(cov_type = 'robust')

    return m2s, m2s_q


Estimating fuzzy RDD (step by step)

In [22]:
fs_k = smf.ols(f"{treatment} ~ I({variable} > {threshold}) + I({variable} - {threshold}) + I({variable} - {threshold}):I({variable} > {threshold})", 
                    data = data[(data[variable] - threshold).apply(abs) < h_opt]).fit()

In [23]:
rf_k = smf.ols(f"{outcome} ~ I({variable} > {threshold}) + I({variable} - {threshold}) + I({variable} - {threshold}):I({variable} > {threshold})", 
                    data = data[(data[variable] - threshold).apply(abs) < h_opt]).fit()

In [24]:
rf_k.params['I(running > 2.75)[T.True]']

-0.1466697915712528

In [25]:
fs_k.params['I(running > 2.75)[T.True]']

-0.9098910568804895

In [26]:
rdd_est = rf_k.params['I(running > 2.75)[T.True]'] / fs_k.params['I(running > 2.75)[T.True]']

In [27]:
rdd_est

0.16119489301730466

Checking quadratic specification, to remove potential biases

In [28]:
fs_q = smf.ols(f"{treatment} ~ I({variable} > {threshold}) + I({variable} - {threshold}) + I(({variable} - {threshold}) ** 2) + I({variable} - {threshold}):I({variable} > {threshold})+ I(({variable} - {threshold})**2):I({variable} > {threshold})", 
                    data = data[(data[variable] - threshold).apply(abs) < h_opt]).fit()

In [29]:
rf_q = smf.ols(f"{outcome} ~ I({variable} > {threshold}) + I({variable} - {threshold}) + I(({variable} - {threshold}) ** 2) + I({variable} - {threshold}):I({variable} > {threshold})+ I(({variable} - {threshold})**2):I({variable} > {threshold})", 
                    data = data[(data[variable] - threshold).apply(abs) < h_opt]).fit()

In [30]:
rdd_est_q = rf_q.params['I(running > 2.75)[T.True]'] / fs_q.params['I(running > 2.75)[T.True]']

In [31]:
fs_q.params['I(running > 2.75)[T.True]']

-0.8942214965870494

In [32]:
rf_q.params['I(running > 2.75)[T.True]']

-0.15802655577353614

In [33]:
rdd_est

0.16119489301730466

Implementation

In [34]:
def get_results(model, treatment, variable, threshold):

    return (model.params[treatment], 
            model.std_errors[treatment], 
            model.first_stage.individual[treatment].params[f'I({variable} > {threshold})'])

In [35]:
def get_graph(data, variable, threshold, h_opt, variable_check, label):

    
    # Graph for linear - Edge
    sns.regplot(data = data[(data[variable] >= threshold - h_opt) & (data[variable] <= threshold)], x = variable, y = variable_check, x_bins = 5, order = 1, color = 'red')
    sns.regplot(data = data[(data[variable] >= threshold - h_opt) & (data[variable] <= threshold)], x = variable, y = variable_check, x_bins = 5, order = 2, color = 'gray')
    
    sns.regplot(data = data[(data[variable] <= threshold + h_opt) & (data[variable] > threshold)], x = variable, y = variable_check, x_bins = 5, order = 1, color = 'red')
    sns.regplot(data = data[(data[variable] <= threshold + h_opt) & (data[variable] > threshold)], x = variable, y = variable_check, x_bins = 5, order = 2, color = 'gray')

    plt.axvline(threshold)
    
    plt.savefig(f"graph_rdd_{label}.png")

    plt.close()
    
    




In [36]:
def get_estimates(data, variable, treatment, outcome, threshold):
    h_opt_edge, h_opt_unif = get_optimal_bandwith(data, variable, outcome, threshold)    
    
    l_edge, q_edge = estimate_frdd(data, variable, treatment, outcome, h_opt_edge, threshold)
    
    l_unif, q_unif = estimate_frdd(data, variable, treatment, outcome, h_opt_unif, threshold)
    
    estimates = pd.DataFrame(map(lambda model: get_results(model, treatment, variable, threshold), 
                                 [l_unif, q_unif, l_edge, q_edge]), 
                             columns = ['estimate', 'se', 'first_stage'], 
                             index = ['Linear - Uniform Kernel', 'Quadratic - Uniform Kernel',
                                      'Linear - Edge Kernel', 'Quadratic - Edge Kernel']
                            )

    estimates['reduce_form'] = estimates['estimate']/estimates['first_stage']
    
    get_graph(data, variable, threshold, h_opt_edge, treatment, f"{variable}_{treatment}_edge")
    get_graph(data, variable, threshold, h_opt_edge, outcome, f"{variable}_{outcome}_edge")

    get_graph(data, variable, threshold, h_opt_unif, treatment, f"{variable}_{treatment}_uni")
    get_graph(data, variable, threshold, h_opt_unif, outcome, f"{variable}_{outcome}_uni")
    
    
                                                                                                                                                            
    return estimates

In [37]:
get_estimates(data, variable, treatment, "male", threshold)

,estimate,se,first_stage,reduce_form
Linear - Uniform Kernel,-0.007019,0.010525,-0.905595,0.007750
Quadratic - Uniform Kernel,-0.004538,0.016543,-0.892522,0.005084
Linear - Edge Kernel,-0.002185,0.013814,-0.900449,0.002426
Quadratic - Edge Kernel,0.004328,0.022436,-0.873976,-0.004952


In [38]:
get_estimates(data, variable, treatment, "black", threshold)

,estimate,se,first_stage,reduce_form
Linear - Uniform Kernel,-0.016666,0.009506,-0.904507,0.018426
Quadratic - Uniform Kernel,-0.027412,0.015104,-0.889512,0.030817
Linear - Edge Kernel,-0.031750,0.012868,-0.893220,0.035546
Quadratic - Edge Kernel,-0.027956,0.021832,-0.878667,0.031816


In [39]:
get_estimates(data, variable, treatment, outcome, threshold)

,estimate,se,first_stage,reduce_form
Linear - Uniform Kernel,0.161195,0.012959,-0.909891,-0.177158
Quadratic - Uniform Kernel,0.176720,0.019817,-0.894221,-0.197624
Linear - Edge Kernel,0.148413,0.016470,-0.901659,-0.164600
Quadratic - Edge Kernel,0.276184,0.026063,-0.887835,-0.311075


c)

Since the estimates we get should be interpreted as local, and in this case our instrument is the math score, we should considere this effecct as the effect on compliers under this specification, students who will attend summer school if they got less than 2.75 in math scores compared to students who wouldn't atend summer school is they got more than 2.75. Since there is a chance of students of still enroll in summer school because of the reading test, this result is not comparable with the one we got in the previous specification.

In [40]:
get_estimates(data, "mtge0", treatment, outcome, threshold)

,estimate,se,first_stage,reduce_form
Linear - Uniform Kernel,0.185609,0.043368,-0.294229,-0.630831
Quadratic - Uniform Kernel,0.128578,0.069976,-0.278468,-0.461733
Linear - Edge Kernel,0.139615,0.056620,-0.282707,-0.493848
Quadratic - Edge Kernel,0.131916,0.089704,-0.285597,-0.461895


d)

Since the estimates we get should be interpreted as local, and since in this case our instrument is the reading score, we should considere this effect as the effect on compliers under this specification, students who will attend summer school if they got less than 2.75 in the reading scores compared to students who wouldn't atend summer school is they got more than 2.75 in this same score. Since there is a chance these students are still enrolling in summer school because of the math test, this result is not comparable to the one we got in the previous specification.

In [41]:
get_estimates(data, "rdge0", treatment, outcome, threshold)

,estimate,se,first_stage,reduce_form
Linear - Uniform Kernel,0.120670,0.018835,-0.728940,-0.165541
Quadratic - Uniform Kernel,0.147400,0.029578,-0.707738,-0.208269
Linear - Edge Kernel,0.129178,0.023489,-0.714479,-0.180800
Quadratic - Edge Kernel,0.199581,0.036976,-0.709273,-0.281389


e)

In [42]:
import numpy as np
import pandas as pd
from scipy import stats, optimize
from scipy.sparse import csc_matrix
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
import warnings
from typing import Optional, Union, Tuple, Dict, Any

class RDHonest:
    """
    Python implementation of RDHonest package for honest inference in regression discontinuity designs.
    
    This class implements bias-aware confidence intervals for sharp and fuzzy RD designs
    based on Armstrong and Kolesár (2018, 2020) and Kolesár and Rothe (2018).
    """
    
    def __init__(self):
        self.results_ = None
        self.data_ = None
        
    def fit(self, 
            formula: str = None,
            data: pd.DataFrame = None,
            y: np.ndarray = None,
            x: np.ndarray = None,
            d: np.ndarray = None,
            cutoff: float = 0,
            M: Union[float, list] = None,
            kernel: str = "triangular",
            opt_criterion: str = "MSE",
            h: Optional[float] = None,
            se_method: str = "nn",
            alpha: float = 0.05,
            beta: float = 0.8,
            sclass: str = "H",
            T0: float = 0,
            point_inference: bool = False,
            weights: Optional[np.ndarray] = None,
            subset: Optional[np.ndarray] = None,
            fuzzy: bool = False) -> 'RDHonest':
        """
        Fit RD model with honest confidence intervals.
        
        Parameters:
        -----------
        formula : str, optional
            Formula in format "y ~ x" for sharp RD or "y | d ~ x" for fuzzy RD
        data : pd.DataFrame, optional
            DataFrame containing the variables
        y : np.ndarray, optional
            Outcome variable
        x : np.ndarray, optional
            Running variable
        d : np.ndarray, optional
            Treatment variable (for fuzzy RD)
        cutoff : float, default=0
            RD cutoff value
        M : float or list
            Smoothness parameter(s)
        kernel : str, default="triangular"
            Kernel type: "triangular", "uniform", or "optimal"
        opt_criterion : str, default="MSE"
            Optimization criterion: "MSE", "FLCI", or "OCI"
        h : float, optional
            Bandwidth (if None, optimal bandwidth is computed)
        se_method : str, default="nn"
            Standard error method: "nn", "EHW", or "supplied"
        alpha : float, default=0.05
            Significance level
        sclass : str, default="H"
            Smoothness class: "H" (Hölder) or "T" (Taylor)
        point_inference : bool, default=False
            Whether to perform inference at a point
        weights : np.ndarray, optional
            Observation weights
        subset : np.ndarray, optional
            Boolean array for subsetting data
        fuzzy : bool, default=False
            Whether this is a fuzzy RD design
        """
        
        # Parse input data
        if formula and data is not None:
            y, x, d, fuzzy = self._parse_formula(formula, data)
        elif y is None or x is None:
            raise ValueError("Must provide either formula+data or y+x arrays")
            
        # Apply subset if provided
        if subset is not None:
            y, x = y[subset], x[subset]
            if d is not None:
                d = d[subset]
            if weights is not None:
                weights = weights[subset]
        
        # Center running variable at cutoff
        x_centered = x - cutoff
        
        # Validate inputs
        self._validate_inputs(y, x_centered, d, M, fuzzy)
        
        # Store data
        self.data_ = {
            'y': y, 'x': x_centered, 'd': d, 'cutoff': cutoff,
            'weights': weights, 'fuzzy': fuzzy
        }
        
        # Estimate preliminary variance if needed
        if se_method in ["nn", "EHW"] or h is None:
            sigma2_est = self._estimate_variance(y, x_centered, d, fuzzy)
        else:
            sigma2_est = None
            
        # Compute optimal bandwidth if not provided
        if h is None:
            h = self._optimal_bandwidth(y, x_centered, d, M, kernel, 
                                      opt_criterion, sigma2_est, sclass, fuzzy)
        
        # Fit the model
        results = self._fit_rd_model(y, x_centered, d, h, M, kernel, 
                                   se_method, alpha, sclass, T0, 
                                   point_inference, weights, fuzzy, sigma2_est)
        
        self.results_ = results
        return self
    
    def _parse_formula(self, formula: str, data: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray, np.ndarray, bool]:
        """Parse R-style formula"""
        fuzzy = '|' in formula
        
        if fuzzy:
            # Fuzzy RD: "y | d ~ x"
            left, right = formula.split('~')
            y_name, d_name = [v.strip() for v in left.split('|')]
            x_name = right.strip()
            
            y = data[y_name].values
            x = data[x_name].values
            d = data[d_name].values
        else:
            # Sharp RD: "y ~ x"
            y_name, x_name = [v.strip() for v in formula.split('~')]
            y = data[y_name].values
            x = data[x_name].values
            d = None
            
        return y, x, d, fuzzy
    
    def _validate_inputs(self, y: np.ndarray, x: np.ndarray, d: np.ndarray, 
                        M: Union[float, list], fuzzy: bool):
        """Validate input parameters"""
        if len(y) != len(x):
            raise ValueError("y and x must have same length")
            
        if fuzzy and (d is None or len(d) != len(y)):
            raise ValueError("For fuzzy RD, d must be provided with same length as y")
            
        if M is None:
            raise ValueError("Smoothness parameter M must be provided")
            
        if fuzzy and not isinstance(M, (list, np.ndarray)):
            raise ValueError("For fuzzy RD, M must be a list/array of length 2")
            
        if fuzzy and len(M) != 2:
            raise ValueError("For fuzzy RD, M must have length 2 [M_Y, M_D]")
    
    def _estimate_variance(self, y: np.ndarray, x: np.ndarray, d: np.ndarray, 
                          fuzzy: bool) -> Dict[str, float]:
        """Estimate preliminary variance using rule-of-thumb bandwidth"""
        n = len(y)
        
        # Silverman's rule of thumb for bandwidth
        h_rot = 1.06 * np.std(x) * n**(-1/5)
        
        # Fit local linear regression on both sides
        sigma2_y_left = self._local_variance_estimate(y[x <= 0], x[x <= 0], h_rot)
        sigma2_y_right = self._local_variance_estimate(y[x > 0], x[x > 0], h_rot)
        
        result = {
            'sigma2_y_left': sigma2_y_left,
            'sigma2_y_right': sigma2_y_right
        }
        
        if fuzzy:
            sigma2_d_left = self._local_variance_estimate(d[x <= 0], x[x <= 0], h_rot)
            sigma2_d_right = self._local_variance_estimate(d[x > 0], x[x > 0], h_rot)
            
            # Estimate covariance
            cov_yd_left = self._local_covariance_estimate(
                y[x <= 0], d[x <= 0], x[x <= 0], h_rot)
            cov_yd_right = self._local_covariance_estimate(
                y[x > 0], d[x > 0], x[x > 0], h_rot)
            
            result.update({
                'sigma2_d_left': sigma2_d_left,
                'sigma2_d_right': sigma2_d_right,
                'cov_yd_left': cov_yd_left,
                'cov_yd_right': cov_yd_right
            })
        
        return result
    
    def _local_variance_estimate(self, y: np.ndarray, x: np.ndarray, h: float) -> float:
        """Estimate local variance using local linear regression"""
        if len(y) < 3:
            return np.var(y) if len(y) > 0 else 1.0
            
        try:
            # Fit local linear regression
            X = np.column_stack([np.ones(len(x)), x])
            reg = LinearRegression().fit(X, y)
            residuals = y - reg.predict(X)
            return np.var(residuals)
        except:
            return np.var(y) if len(y) > 0 else 1.0
    
    def _local_covariance_estimate(self, y: np.ndarray, d: np.ndarray, 
                                  x: np.ndarray, h: float) -> float:
        """Estimate local covariance"""
        if len(y) < 3:
            return np.cov(y, d)[0, 1] if len(y) > 1 else 0.0
            
        try:
            # Fit local linear regressions for both y and d
            X = np.column_stack([np.ones(len(x)), x])
            reg_y = LinearRegression().fit(X, y)
            reg_d = LinearRegression().fit(X, d)
            
            resid_y = y - reg_y.predict(X)
            resid_d = d - reg_d.predict(X)
            
            return np.cov(resid_y, resid_d)[0, 1]
        except:
            return np.cov(y, d)[0, 1] if len(y) > 1 else 0.0
    
    def _optimal_bandwidth(self, y: np.ndarray, x: np.ndarray, d: np.ndarray,
                          M: Union[float, list], kernel: str, opt_criterion: str,
                          sigma2_est: Dict[str, float], sclass: str, 
                          fuzzy: bool) -> float:
        """Compute optimal bandwidth"""
        n = len(y)
        
        # Simple bandwidth selection based on rule of thumb
        # This is a simplified version - full implementation would involve
        # more complex optimization as in the original R package
        
        if kernel == "uniform":
            h_rot = 1.84 * np.std(x) * n**(-1/5)
        else:  # triangular
            h_rot = 2.78 * np.std(x) * n**(-1/5)
            
        # Apply smoothness constraint
        M_eff = M if not fuzzy else max(M)
        
        # Adjust based on optimization criterion
        if opt_criterion == "FLCI":
            h_rot *= 1.1  # Slightly larger for CI length optimization
        elif opt_criterion == "OCI":
            h_rot *= 0.9  # Slightly smaller for one-sided CI
            
        return max(h_rot, np.std(x) / n)  # Ensure minimum bandwidth
    
    def _kernel_weights(self, u: np.ndarray, kernel: str) -> np.ndarray:
        """Compute kernel weights"""
        if kernel == "uniform":
            return (np.abs(u) <= 1).astype(float)
        elif kernel == "triangular":
            return np.maximum(1 - np.abs(u), 0)
        elif kernel == "optimal":
            # Simplified optimal kernel (full implementation more complex)
            return np.maximum(1 - np.abs(u), 0)
        else:
            raise ValueError(f"Unknown kernel: {kernel}")
    
    def _fit_rd_model(self, y: np.ndarray, x: np.ndarray, d: np.ndarray,
                     h: float, M: Union[float, list], kernel: str,
                     se_method: str, alpha: float, sclass: str, T0: float,
                     point_inference: bool, weights: Optional[np.ndarray],
                     fuzzy: bool, sigma2_est: Dict[str, float]) -> Dict[str, Any]:
        """Fit the RD model and compute confidence intervals"""
        
        n = len(y)
        
        # Create design matrix for local linear regression
        # Left side of cutoff
        left_mask = x <= 0
        x_left = x[left_mask]
        y_left = y[left_mask]
        
        # Right side of cutoff  
        right_mask = x > 0
        x_right = x[right_mask]
        y_right = y[right_mask]
        
        if fuzzy:
            d_left = d[left_mask]
            d_right = d[right_mask]
            
        # Compute kernel weights
        w_left = self._kernel_weights(x_left / h, kernel)
        w_right = self._kernel_weights(x_right / h, kernel)
        
        if weights is not None:
            w_left *= weights[left_mask]
            w_right *= weights[right_mask]
        
        # Fit local linear regressions
        if not fuzzy:
            # Sharp RD
            estimate, se, bias_bound = self._sharp_rd_estimation(
                y_left, x_left, w_left, y_right, x_right, w_right,
                h, M, se_method, sclass)
        else:
            # Fuzzy RD
            estimate, se, bias_bound = self._fuzzy_rd_estimation(
                y_left, x_left, d_left, w_left, y_right, x_right, d_right, w_right,
                h, M, se_method, sclass, T0)
        
        # Compute honest confidence intervals
        ci_lower, ci_upper = self._honest_ci(estimate, se, bias_bound, alpha)
        
        # Prepare results
        results = {
            'estimate': estimate,
            'se': se,
            'bias_bound': bias_bound,
            'ci_lower': ci_lower,
            'ci_upper': ci_upper,
            'bandwidth': h,
            'n_left': len(y_left),
            'n_right': len(y_right),
            'alpha': alpha,
            'kernel': kernel,
            'M': M,
            'sclass': sclass
        }
        
        return results
    
    def _sharp_rd_estimation(self, y_left: np.ndarray, x_left: np.ndarray, w_left: np.ndarray,
                            y_right: np.ndarray, x_right: np.ndarray, w_right: np.ndarray,
                            h: float, M: float, se_method: str, sclass: str) -> Tuple[float, float, float]:
        """Estimate sharp RD parameter"""
        
        # Fit local linear regression on left side
        if len(x_left) > 1:
            X_left = np.column_stack([np.ones(len(x_left)), x_left])
            reg_left = LinearRegression().fit(X_left, y_left, sample_weight=w_left)
            y_left_0 = reg_left.predict([[1, 0]])[0]
        else:
            y_left_0 = np.mean(y_left) if len(y_left) > 0 else 0
        
        # Fit local linear regression on right side
        if len(x_right) > 1:
            X_right = np.column_stack([np.ones(len(x_right)), x_right])
            reg_right = LinearRegression().fit(X_right, y_right, sample_weight=w_right)
            y_right_0 = reg_right.predict([[1, 0]])[0]
        else:
            y_right_0 = np.mean(y_right) if len(y_right) > 0 else 0
        
        # RD estimate
        estimate = y_right_0 - y_left_0
        
        # Compute standard error
        se = self._compute_se_sharp(y_left, x_left, w_left, y_right, x_right, w_right, se_method)
        
        # Compute bias bound
        bias_bound = self._compute_bias_bound(h, M, sclass)
        
        return estimate, se, bias_bound
    
    def _fuzzy_rd_estimation(self, y_left: np.ndarray, x_left: np.ndarray, d_left: np.ndarray, w_left: np.ndarray,
                            y_right: np.ndarray, x_right: np.ndarray, d_right: np.ndarray, w_right: np.ndarray,
                            h: float, M: list, se_method: str, sclass: str, T0: float) -> Tuple[float, float, float]:
        """Estimate fuzzy RD parameter using 2SLS"""
        
        # First stage: effect on treatment
        d_left_0 = np.mean(d_left) if len(d_left) > 0 else 0
        d_right_0 = np.mean(d_right) if len(d_right) > 0 else 0
        first_stage = d_right_0 - d_left_0
        
        # Reduced form: effect on outcome
        y_left_0 = np.mean(y_left) if len(y_left) > 0 else 0
        y_right_0 = np.mean(y_right) if len(y_right) > 0 else 0
        reduced_form = y_right_0 - y_left_0
        
        # Fuzzy RD estimate (Wald ratio)
        if abs(first_stage) < 1e-8:
            estimate = 0
            warnings.warn("First stage is close to zero. Fuzzy RD may not be identified.")
        else:
            estimate = reduced_form / first_stage
        
        # Compute standard error (simplified)
        se_rf = self._compute_se_sharp(y_left, x_left, w_left, y_right, x_right, w_right, se_method)
        se_fs = self._compute_se_sharp(d_left, x_left, w_left, d_right, x_right, w_right, se_method)
        
        # Delta method approximation
        se = np.sqrt((se_rf/first_stage)**2 + (reduced_form * se_fs / first_stage**2)**2)
        
        # Bias bound for fuzzy RD
        bias_bound = self._compute_bias_bound_fuzzy(h, M, sclass, abs(estimate), first_stage)
        
        return estimate, se, bias_bound
    
    def _compute_se_sharp(self, y_left: np.ndarray, x_left: np.ndarray, w_left: np.ndarray,
                         y_right: np.ndarray, x_right: np.ndarray, w_right: np.ndarray,
                         se_method: str) -> float:
        """Compute standard errors for sharp RD"""
        
        # Simplified nearest neighbor variance estimator
        var_left = np.var(y_left) / len(y_left) if len(y_left) > 0 else 0
        var_right = np.var(y_right) / len(y_right) if len(y_right) > 0 else 0
        
        return np.sqrt(var_left + var_right)
    
    def _compute_bias_bound(self, h: float, M: float, sclass: str) -> float:
        """Compute bias bound"""
        if sclass == "H":  # Hölder
            return M * h**2 / 2
        else:  # Taylor
            return M * h**2
    
    def _compute_bias_bound_fuzzy(self, h: float, M: list, sclass: str, 
                                 estimate: float, first_stage: float) -> float:
        """Compute bias bound for fuzzy RD"""
        M_y, M_d = M
        
        if sclass == "H":
            bias_y = M_y * h**2 / 2
            bias_d = M_d * h**2 / 2
        else:
            bias_y = M_y * h**2
            bias_d = M_d * h**2
        
        # Combined bias bound using delta method
        return abs(bias_y / first_stage) + abs(estimate * bias_d / first_stage)
    
    def _honest_ci(self, estimate: float, se: float, bias_bound: float, 
                   alpha: float) -> Tuple[float, float]:
        """Compute honest confidence intervals"""
        
        # Critical value accounting for bias
        if bias_bound == 0:
            cv = stats.norm.ppf(1 - alpha/2)
        else:
            # Use bias-aware critical value (simplified)
            cv = stats.norm.ppf(1 - alpha/2) + bias_bound / se
        
        half_width = cv * se
        
        return estimate - half_width, estimate + half_width
    
    def summary(self) -> pd.DataFrame:
        """Return summary of results"""
        if self.results_ is None:
            raise ValueError("Model has not been fitted yet")
            
        results = self.results_
        
        summary_data = {
            'Estimate': [results['estimate']],
            'Std. Error': [results['se']],
            'Bias Bound': [results['bias_bound']],
            'CI Lower': [results['ci_lower']],
            'CI Upper': [results['ci_upper']],
            'Bandwidth': [results['bandwidth']],
            'N Left': [results['n_left']],
            'N Right': [results['n_right']]
        }
        
        return pd.DataFrame(summary_data)
    
    def print_results(self):
        """Print formatted results"""
        if self.results_ is None:
            raise ValueError("Model has not been fitted yet")
            
        results = self.results_
        
        print("RD Honest Inference Results")
        print("=" * 40)
        print(f"Estimate: {results['estimate']:.4f}")
        print(f"Standard Error: {results['se']:.4f}")
        print(f"Bias Bound: {results['bias_bound']:.4f}")
        print(f"95% Confidence Interval: [{results['ci_lower']:.4f}, {results['ci_upper']:.4f}]")
        print(f"Bandwidth: {results['bandwidth']:.4f}")
        print(f"Observations: {results['n_left']} (left), {results['n_right']} (right)")
        print(f"Kernel: {results['kernel']}")
        print(f"Smoothness class: {results['sclass']}")
        print(f"Smoothness parameter M: {results['M']}")

In [43]:
variable = 'running'

In [ ]:
data[

In [52]:
# Fit RD model
rd = RDHonest()
rd.fit(formula=f" {outcome} | {treatment} ~ {variable}", data=data, M=[0, 0], cutoff = 2.75, h = 0.05)
rd.print_results()


RD Honest Inference Results
Estimate: -1.1108
Standard Error: 0.0070
Bias Bound: 0.0000
95% Confidence Interval: [-1.1245, -1.0971]
Bandwidth: 0.0500
Observations: 29900 (left), 40931 (right)
Kernel: triangular
Smoothness class: H
Smoothness parameter M: [0, 0]


In [53]:
# Fit RD model
rd = RDHonest()
rd.fit(formula=f" {outcome} | {treatment} ~ {variable}", data=data, M=[0.5, 0.5], cutoff = 2.75, h=0.001)
rd.print_results()


RD Honest Inference Results
Estimate: -1.1108
Standard Error: 0.0070
Bias Bound: 0.0000
95% Confidence Interval: [-1.1245, -1.0971]
Bandwidth: 0.0010
Observations: 29900 (left), 40931 (right)
Kernel: triangular
Smoothness class: H
Smoothness parameter M: [0.5, 0.5]


In [ ]:
# Fit RD model
rd = RDHonest()
rd.fit(formula=f" {outcome} ~ [{summer} ~ {variable}]", data=data, M=0.1)
rd.print_results()